In [2]:
import pandas as pd
import psycopg2
from moexalgo import Ticker
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. ПОДКЛЮЧЕНИЕ К БАЗЕ ДАННЫХ
# ============================================
conn = psycopg2.connect(
    host="109.196.102.91",
    database="default_db",
    user="maksv",
    password="p27Oqwp4Y,X^O^"
)
cursor = conn.cursor()
print("✅ Подключение к БД установлено")

# ============================================
# 2. ЗАГРУЗКА СПИСКА ТИКЕРОВ ИЗ EXCEL-ФАЙЛА
# ============================================
file_url = "https://github.com/maksim200108-bit/FinCore/raw/main/Tickers_indexes.xlsx"

try:
    print(f"\n📥 Загружаем файл с GitHub: {file_url}")
    df_tickers = pd.read_excel(file_url)
    print(f"✅ Файл загружен. Найдено записей: {len(df_tickers)}")
    
    if 'Ticker' not in df_tickers.columns or 'Description' not in df_tickers.columns:
        raise ValueError("В файле отсутствуют колонки 'Ticker' или 'Description'")
    
    # Создаем словари для соответствий
    ticker_to_description = dict(zip(df_tickers['Ticker'], df_tickers['Description']))
    description_to_ticker = dict(zip(df_tickers['Description'], df_tickers['Ticker']))
    
    tickers_list = df_tickers['Ticker'].tolist()
    print(f"\n📊 Список тикеров для обновления ({len(tickers_list)} шт.)")
    
except Exception as e:
    print(f"❌ Ошибка при загрузке файла с GitHub: {e}")
    cursor.close()
    conn.close()
    exit()

# ============================================
# 3. ПОЛУЧЕНИЕ ПОСЛЕДНИХ ДАТ ИЗ БД
# ============================================
print("\n" + "="*50)
print("🔍 АНАЛИЗ ТЕКУЩИХ ДАННЫХ В БД")
print("="*50)

# Получаем максимальную дату для каждого описания
cursor.execute("""
    SELECT description, MAX(date) as last_date
    FROM indexes
    WHERE description IS NOT NULL
    GROUP BY description
    ORDER BY description;
""")
db_last_dates = dict(cursor.fetchall())

print(f"\n📅 Найдено записей в БД: {len(db_last_dates)} описаний")

# ============================================
# 4. ОПРЕДЕЛЕНИЕ ПЕРИОДА ДЛЯ ОБНОВЛЕНИЯ
# ============================================
print("\n" + "="*50)
print("📅 ОПРЕДЕЛЕНИЕ ПЕРИОДА ОБНОВЛЕНИЯ")
print("="*50)

# По умолчанию обновляем за последние 30 дней
days_to_update = 30
end_date = datetime.now()

# Проверяем, есть ли в БД данные за последние дни
cursor.execute("""
    SELECT MAX(date) FROM indexes;
""")
max_date_in_db = cursor.fetchone()[0]

if max_date_in_db:
    max_date = datetime.strptime(str(max_date_in_db), '%Y-%m-%d')
    days_since_last = (end_date - max_date).days
    
    print(f"\n📅 Последняя дата в БД: {max_date_in_db}")
    print(f"📅 Прошло дней: {days_since_last}")
    
    if days_since_last < 30:
        # Если данные свежие, обновляем только с последней даты
        start_date = max_date - timedelta(days=5)  # Берем с запасом в 5 дней
        print(f"🔄 Будет выполнено обновление (update) существующих записей")
    else:
        # Если данных давно нет, загружаем за последние 30 дней
        start_date = end_date - timedelta(days=30)
        print(f"🔄 Будет выполнена догрузка (insert) новых записей")
else:
    # Если в БД нет данных, загружаем за всё время
    start_date = datetime(2010, 1, 1)
    print(f"🔄 БД пуста. Будет выполнена полная загрузка")

print(f"📅 Период загрузки: {start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')}")

# ============================================
# 5. ЗАГРУЗКА НОВЫХ ДАННЫХ
# ============================================
print("\n" + "="*50)
print("📥 ЗАГРУЗКА НОВЫХ ДАННЫХ")
print("="*50)

all_new_data = []
updated_tickers = []
failed_tickers = []

for ticker in tickers_list:
    try:
        print(f"\n🔄 Обрабатываем индекс: {ticker}")
        
        description = ticker_to_description.get(ticker, '')
        print(f"  📝 Описание: {description[:50]}...")
        
        # Создаем объект для работы с тикером
        index = Ticker(ticker)
        
        # Загружаем данные за нужный период
        df = index.candles(
            start=start_date.strftime('%Y-%m-%d'),
            end=end_date.strftime('%Y-%m-%d'),
            period='1D'
        )
        
        if df is not None and not df.empty:
            # Стандартизируем данные
            required_cols = ['begin', 'open', 'close', 'high', 'low', 'value']
            available_cols = [col for col in required_cols if col in df.columns]
            
            if available_cols:
                df_selected = df[available_cols].copy()
                df_selected['description'] = description
                df_selected['ticker'] = ticker
                
                # Удаляем дубликаты и сортируем
                df_selected = df_selected.drop_duplicates(subset=['begin'])
                df_selected = df_selected.sort_values('begin')
                
                all_new_data.append(df_selected)
                updated_tickers.append(ticker)
                
                # Показываем статистику
                last_date = df_selected['begin'].iloc[-1]
                new_records = len(df_selected)
                
                # Проверяем, сколько новых записей
                if description in db_last_dates:
                    old_last = db_last_dates[description]
                    new_after_old = len(df_selected[df_selected['begin'] > old_last])
                    print(f"  ✅ Загружено: {new_records} записей (новых: {new_after_old})")
                else:
                    print(f"  ✅ Загружено: {new_records} записей (новый индекс)")
            else:
                print(f"  ⚠️ Нет нужных колонок. Доступны: {list(df.columns)}")
                failed_tickers.append(ticker)
        else:
            print(f"  ⚠️ Нет данных за период")
            failed_tickers.append(ticker)
            
    except Exception as e:
        print(f"  ❌ Ошибка: {e}")
        failed_tickers.append(ticker)

# ============================================
# 6. ОБЪЕДИНЕНИЕ ДАННЫХ
# ============================================
print("\n" + "="*50)
print("📊 ИТОГИ ЗАГРУЗКИ")
print("="*50)

print(f"\n✅ Успешно обновлено тикеров: {len(updated_tickers)}")
print(f"❌ Не удалось обновить: {len(failed_tickers)}")

if not all_new_data:
    print("\n❌ Нет новых данных для загрузки")
    cursor.close()
    conn.close()
    exit()

final_df = pd.concat(all_new_data, ignore_index=True)
print(f"\n📊 Всего загружено записей: {len(final_df)}")

# ============================================
# 7. ОБНОВЛЕНИЕ БАЗЫ ДАННЫХ
# ============================================
print("\n" + "="*50)
print("💾 ОБНОВЛЕНИЕ БАЗЫ ДАННЫХ")
print("="*50)

insert_count = 0
update_count = 0
error_count = 0

# Создаем временную таблицу для более эффективного обновления
try:
    print("\n🔄 Создаем временную таблицу...")
    cursor.execute("""
        CREATE TEMP TABLE temp_indexes (
            date DATE,
            open NUMERIC,
            close NUMERIC,
            high NUMERIC,
            low NUMERIC,
            value NUMERIC,
            description VARCHAR(100),
            ticker VARCHAR(20)
        );
    """)
    
    # Загружаем данные во временную таблицу
    print("📤 Загружаем данные во временную таблицу...")
    batch_size = 1000
    data_batch = []
    
    for idx, row in final_df.iterrows():
        try:
            # Преобразуем дату
            if hasattr(row['begin'], 'strftime'):
                date_val = row['begin'].strftime('%Y-%m-%d')
            else:
                date_val = pd.to_datetime(row['begin']).strftime('%Y-%m-%d')
            
            data_batch.append((
                date_val,
                float(row['open']) if pd.notna(row['open']) else None,
                float(row['close']) if pd.notna(row['close']) else None,
                float(row['high']) if pd.notna(row['high']) else None,
                float(row['low']) if pd.notna(row['low']) else None,
                float(row['value']) if pd.notna(row['value']) else None,
                row['description'],
                row['ticker']
            ))
            
            # Вставляем батчами
            if len(data_batch) >= batch_size:
                cursor.executemany("""
                    INSERT INTO temp_indexes (date, open, close, high, low, value, description, ticker)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
                """, data_batch)
                data_batch = []
                print(f"    Загружено {idx + 1} записей...")
                
        except Exception as e:
            error_count += 1
    
    # Вставляем оставшиеся записи
    if data_batch:
        cursor.executemany("""
            INSERT INTO temp_indexes (date, open, close, high, low, value, description, ticker)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s);
        """, data_batch)
    
    conn.commit()
    print(f"✅ Данные загружены во временную таблицу")
    
    # Обновляем существующие записи
    print("\n🔄 Обновляем существующие записи...")
    cursor.execute("""
        UPDATE indexes i
        SET 
            open = t.open,
            close = t.close,
            high = t.high,
            low = t.low,
            value = t.value,
            ticker = COALESCE(i.ticker, t.ticker)
        FROM temp_indexes t
        WHERE i.date = t.date 
          AND i.description = t.description;
    """)
    update_count = cursor.rowcount
    print(f"✅ Обновлено записей: {update_count}")
    
    # Вставляем новые записи
    print("\n➕ Вставляем новые записи...")
    cursor.execute("""
        INSERT INTO indexes (date, open, close, high, low, value, description, ticker)
        SELECT t.date, t.open, t.close, t.high, t.low, t.value, t.description, t.ticker
        FROM temp_indexes t
        LEFT JOIN indexes i ON i.date = t.date AND i.description = t.description
        WHERE i.date IS NULL;
    """)
    insert_count = cursor.rowcount
    print(f"✅ Вставлено новых записей: {insert_count}")
    
    # Очищаем временную таблицу
    cursor.execute("DROP TABLE temp_indexes;")
    
except Exception as e:
    print(f"❌ Ошибка при обновлении БД: {e}")
    conn.rollback()
finally:
    conn.commit()

print(f"\n📊 Итоги обновления:")
print(f"  • Вставлено новых записей: {insert_count}")
print(f"  • Обновлено существующих: {update_count}")
print(f"  • Ошибок при обработке: {error_count}")

# ============================================
# 8. ПРОВЕРКА РЕЗУЛЬТАТА
# ============================================
print("\n" + "="*50)
print("🔍 ПРОВЕРКА РЕЗУЛЬТАТА")
print("="*50)

# Общая статистика
cursor.execute("SELECT COUNT(*) FROM indexes;")
total_count = cursor.fetchone()[0]
print(f"\n📊 Всего записей в таблице: {total_count}")

# Статистика по обновленным индексам
print("\n📈 Статистика по обновленным индексам:")
cursor.execute("""
    SELECT 
        ticker,
        description,
        COUNT(*) as records,
        MIN(date) as first_date,
        MAX(date) as last_date,
        COUNT(CASE WHEN date >= CURRENT_DATE - INTERVAL '30 days' THEN 1 END) as last_30_days
    FROM indexes
    WHERE ticker = ANY(%s)
    GROUP BY ticker, description
    ORDER BY ticker;
""", (updated_tickers,))

stats_rows = cursor.fetchall()
for row in stats_rows:
    print(f"\n  {row[0]}:")
    print(f"    • Описание: {row[1][:50]}")
    print(f"    • Записей: {row[2]}")
    print(f"    • Период: {row[3]} - {row[4]}")
    print(f"    • За 30 дней: {row[5]}")

# Проверка последних записей
print("\n🔍 Последние 10 записей в БД:")
cursor.execute("""
    SELECT date, ticker, description, open, close
    FROM indexes 
    ORDER BY date DESC
    LIMIT 10;
""")
for row in cursor.fetchall():
    ticker_display = row[1] if row[1] else 'N/A'
    print(f"  {row[0]} | {ticker_display:10} | {row[2][:30]:30} | O:{row[3]:8.2f} C:{row[4]:8.2f}")

# ============================================
# 9. ЗАКРЫТИЕ ПОДКЛЮЧЕНИЯ
# ============================================
cursor.close()
conn.close()
print("\n🔌 Соединение с БД закрыто")
print("✅ Скрипт обновления завершен успешно!")

✅ Подключение к БД установлено

📥 Загружаем файл с GitHub: https://github.com/maksim200108-bit/FinCore/raw/main/Tickers_indexes.xlsx
✅ Файл загружен. Найдено записей: 25

📊 Список тикеров для обновления (25 шт.)

🔍 АНАЛИЗ ТЕКУЩИХ ДАННЫХ В БД

📅 Найдено записей в БД: 25 описаний

📅 ОПРЕДЕЛЕНИЕ ПЕРИОДА ОБНОВЛЕНИЯ

📅 Последняя дата в БД: 2026-02-27
📅 Прошло дней: 2
🔄 Будет выполнено обновление (update) существующих записей
📅 Период загрузки: 2026-02-22 - 2026-03-01

📥 ЗАГРУЗКА НОВЫХ ДАННЫХ

🔄 Обрабатываем индекс: MOEXBMI
  📝 Описание: Индекс акций широкого рынка...
  ✅ Загружено: 4 записей (новый индекс)

🔄 Обрабатываем индекс: MRBC
  📝 Описание: Индекс МосБиржи 15...
  ✅ Загружено: 4 записей (новый индекс)

🔄 Обрабатываем индекс: MOEXBC
  📝 Описание: Индекс голубых фишек...
  ✅ Загружено: 4 записей (новый индекс)

🔄 Обрабатываем индекс: IMOEX
  📝 Описание: Индекс МосБиржи...
  ✅ Загружено: 4 записей (новый индекс)

🔄 Обрабатываем индекс: RTSI
  📝 Описание: Индекс РТС...
  ✅ Загружено: 4 